In [ ]:
import os, subprocess, time, json, re, threading, urllib.request
TOPIC = "__NTFY_TOPIC__"
def notify(msg):
    print(msg, flush=True)
    try:
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{TOPIC}", data=str(msg).encode(), method="POST"), timeout=15)
    except Exception as e:
        print("notify failed", e)
QUANT = "Q4_K_M"
MODEL = f"hf.co/luminenio/gnani-evon-v3.3-30B-A3B-GGUF:{QUANT}"
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
disk = subprocess.run("df -h / /kaggle/working /tmp | tail -n +2", shell=True, capture_output=True, text=True).stdout.strip()
notify(f"STAGE boot | gpu: {gpu} | disk: {disk}")

In [ ]:
r = subprocess.run("apt-get install -y -qq zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/tmp/ollama_install.log 2>&1; "
                   "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared; "
                   "ollama --version; cloudflared --version", shell=True, capture_output=True, text=True)
notify(f"STAGE installed | {r.stdout.strip()} {r.stderr.strip()[-300:]}")

In [ ]:
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="2",
           OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_SCHED_SPREAD="1")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
t = time.time()
r = subprocess.run(f"ollama pull {MODEL} 2>&1 | tail -c 400", shell=True, capture_output=True, text=True)
if r.returncode != 0 or "error" in r.stdout.lower():
    notify(f"STAGE pull-failed {QUANT}: {r.stdout[-300:]} -> trying Q3_K_M")
    QUANT = "Q3_K_M"; MODEL = f"hf.co/luminenio/gnani-evon-v3.3-30B-A3B-GGUF:{QUANT}"
    r = subprocess.run(f"ollama pull {MODEL} 2>&1 | tail -c 400", shell=True, capture_output=True, text=True)
notify(f"STAGE pulled {MODEL} in {time.time()-t:.0f}s | {r.stdout[-200:]}")

In [ ]:
def ask(content, system="Reply only with JSON: {\"customer\": string, \"amount\": number, \"type\": \"credit\"|\"payment\"}"):
    req = urllib.request.Request("http://localhost:11434/v1/chat/completions", method="POST", headers={"Content-Type": "application/json"},
        data=json.dumps({"model": MODEL, "temperature": 0.1, "max_tokens": 200,
                         "messages": [{"role": "system", "content": system}, {"role": "user", "content": content}]}).encode())
    return json.loads(urllib.request.urlopen(req, timeout=900).read())["choices"][0]["message"]["content"]
t = time.time()
try:
    out = ask("Ramesh bhaiya ko dhai sau ka atta udhaar likh do")
    ps = subprocess.run("ollama ps", shell=True, capture_output=True, text=True).stdout
    notify(f"STAGE warm {time.time()-t:.0f}s | {out[:200]} | {ps}")
except Exception as e:
    notify(f"STAGE warm-failed {e} | {open('/kaggle/working/ollama.log').read()[-500:]}")

In [ ]:
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
def drain():
    global URL
    for line in tunnel.stdout:
        m = re.search(r"https://[a-z0-9-]+.trycloudflare.com", line)
        if m and not URL:
            URL = m.group(0)
threading.Thread(target=drain, daemon=True).start()
for _ in range(60):
    if URL: break
    time.sleep(2)
notify(f"URL {URL}/v1 MODEL {MODEL}")

In [ ]:
# Keep alive (Kaggle stops the session at 12 h). Re-announce the URL every 10 minutes.
start = time.time()
while time.time() - start < 11.7 * 3600:
    time.sleep(600)
    alive = ollama.poll() is None and tunnel.poll() is None
    notify(f"URL {URL}/v1 MODEL {MODEL} alive={alive} uptime={(time.time()-start)/3600:.1f}h")